In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/ATPM/Payloads.csv", encoding="latin1")

invalid_rows = []

def has_query_param(url, idx):
    if not isinstance(url, str) or not url.strip():
        return False
    try:
        return bool(urlsplit(url.strip()).query)
    except ValueError:
        invalid_rows.append(idx)
        return False

df["has_query"] = [
    has_query_param(url, idx)
    for idx, url in df["Payloads"].items()
]

print("Has query")
print(df["has_query"].value_counts())
print("Number of invalid URLs:", len(invalid_rows))
print("\nInvalid rows:")
print(df.loc[invalid_rows, "Payloads"].head(20))


Has query
has_query
False    23152
True     20065
Name: count, dtype: int64
Number of invalid URLs: 17

Invalid rows:
26152                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [15]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/ATPM/Payloads.csv", encoding="latin1")

# Validate columns
assert "Payloads" in df.columns, f"Missing column: Payloads. Found: {list(df.columns)}"
assert "Class" in df.columns, f"Missing column: Class. Found: {list(df.columns)}"

# Warn on unknown classes
unknown = df["Payloads"].isna()
if unknown.any():
    print("⚠️ Warning: Unknown Payloads values detected. Unique values:")
    print(df.loc[unknown, "Payloads"].astype(str).str.strip().unique()[:20])

df_clean = df.drop_duplicates(subset=['Payloads']).dropna(subset=['Payloads'])

print(f"Total original rows: {len(df)}")
print(f"Total clean rows: {len(df_clean)}")

# Đếm số lượng từng class
counts = df_clean["Class"].value_counts()

print("\n[+] Class counts:")
print(counts)

# Tính tỷ lệ %
ratios = df_clean["Class"].value_counts(normalize=True) * 100

print("\n[+] Class ratios (%):")
print(ratios)

print("\n[+] Write to CSV")
df_clean.to_csv("/content/drive/MyDrive/ATPM/Payloads_clean.csv")


Total original rows: 43217
Total clean rows: 42671

[+] Class counts:
Class
Benign       28068
Malicious    14603
Name: count, dtype: int64

[+] Class ratios (%):
Class
Benign       65.777694
Malicious    34.222306
Name: proportion, dtype: float64

[+] Write to CSV


In [ ]:
pd.set_option("display.max_columns", None)
df.head()

,Payloads,Class,has_query
0,http://www.nwce.gov.uk/search_process.php?keyw...,Malicious,True
1,http://www.manchester.gov.uk/site/scripts/goog...,Malicious,True
2,http://www.ldsmissions.com/us/index.php?action...,Malicious,True
3,http://education.powys.gov.uk/english/adult_ed...,Malicious,True
4,http://www.northwarks.gov.uk/site/scripts/goog...,Malicious,True


In [17]:

import re, string
import unicodedata
import html
from urllib.parse import unquote_plus, urlsplit, parse_qsl
from typing import List, Dict, Tuple

import pandas as pd

# ---------------- Config ----------------
INPUT_CSV = "/content/drive/MyDrive/ATPM/Payloads_clean.csv"
OUTPUT_CSV = "/content/drive/MyDrive/ATPM/features.csv"
TEXT_COL = "Payloads"
CLASS_COL = "Class"
KEEP_ORIGINAL = True
# ----------------------------------------


# =======================
# Robust CSV loading
# =======================
def read_csv_robust(path: str) -> pd.DataFrame:
    # Try UTF-8 first; fallback to latin1; also skip badly-formed lines if needed.
    try:
        return pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        print("⚠️ UTF-8 decode failed -> fallback to latin1")
        return pd.read_csv(path, encoding="latin1")


# =======================
# Table 1: Structural features
# =======================
PUNCTUATION_CHARS: List[str] = [
    "&", "%", "/", "\\", "+", "'", "?", "!", ";", "#", "=", "[", "]", "$",
    "(", ")", "^", "*", ",", "-", "<", ">", "@", "_", ":", "{", "}", "~", ".",
    " ", "|", "¦", '"',
]

PUNCTUATION_COMBOS: List[str] = ["><", "'\"><", "[]", "==", "&#"]


def safe_feature_name(token: str, prefix: str) -> str:
    mapping = {
        "&": "amp",
        "%": "pct",
        "/": "slash",
        "\\": "bslash",
        "+": "plus",
        "'": "squote",
        '"': "dquote",
        "?": "qmark",
        "!": "emark",
        ";": "semi",
        "#": "hash",
        "=": "eq",
        "[": "lbrack",
        "]": "rbrack",
        "$": "dollar",
        "(": "lparen",
        ")": "rparen",
        "^": "caret",
        "*": "star",
        ",": "comma",
        "-": "dash",
        "<": "lt",
        ">": "gt",
        "@": "at",
        "_": "underscore",
        ":": "colon",
        "{": "lbrace",
        "}": "rbrace",
        "~": "tilde",
        ".": "dot",
        " ": "space",
        "|": "pipe",
        "¦": "brokenbar",
    }
    return f"{prefix}_" + "_".join(mapping.get(ch, f"u{ord(ch)}") for ch in token)


# =======================
# Table 2: Keyword features (word-boundary safe where appropriate)
# =======================

WORD_PATTERNS: List[Tuple[str, re.Pattern]] = [
    # Objects
    ("obj_document", re.compile(r"\bdocument\b")),
    ("obj_window", re.compile(r"\bwindow\b")),
    ("obj_iframe", re.compile(r"\biframe\b")),
    ("obj_location", re.compile(r"\blocation\b")),
    ("obj_this", re.compile(r"\bthis\b")),

    # Events
    ("evt_onload", re.compile(r"\bonload\b")),
    ("evt_onerror", re.compile(r"\bonerror\b")),

    # Methods
    ("mth_createelement", re.compile(r"\bcreateelement\b")),
    ("mth_fromcharcode", re.compile(r"\bstring\.fromcharcode\b")),
    ("mth_search", re.compile(r"\bsearch\b")),

    # Attributes (word boundary helps avoid matching "srcset", etc.)
    ("attr_src", re.compile(r"\bsrc\b")),
    ("attr_href", re.compile(r"\bhref\b")),
    ("attr_cookie", re.compile(r"\bcookie\b")),

    # Reserved
    ("kw_var", re.compile(r"\bvar\b")),

    # Protocol
    ("proto_http", re.compile(r"\bhttp\b")),
]

LITERAL_PATTERNS: List[Tuple[str, str]] = [
    # Tags
    ("tag_div", "<div"),
    ("tag_img", "<img"),
    ("tag_script", "<script"),

    # Functions
    ("fn_eval", "eval("),

    # External file
    ("ext_js", ".js"),
]


# =======================
# Decoding / normalization pipeline
# =======================
BR_TAG_RE = re.compile(r"(?is)<\s*br\s*/?\s*>")

def remove_br_tags(s: str) -> str:
    """Remove injected <br> to reconnect tokens: documen<br>t -> document."""
    if s is None:
        return ""
    return BR_TAG_RE.sub("", str(s))

def decode_layers(s: str, rounds: int = 3) -> str:
    """
    rounds loop to handle multi-encoding:
      - remove <br>
      - URL decode (%xx, '+'->space)
      - HTML unescape (&lt; &amp; &quot; &#xNN;)
      - Unicode normalize (NFKC)
    """
    if s is None:
        return ""
    s = str(s)

    prev = None
    for _ in range(rounds):
        if s == prev:
            break
        prev = s

        s = remove_br_tags(s)

        try:
            s = unquote_plus(s)
        except Exception:
            pass

        try:
            s = html.unescape(s)
        except Exception:
            pass

        try:
            s = unicodedata.normalize("NFKC", s)
        except Exception:
            pass

    return s

# def sanitize_url_for_parse(u: str) -> str:
#     # remove control characters that can break parsing
#     u = re.sub(r"[\x00-\x1F\x7F]+", "", u.strip())
#     return u

def url_to_payload_core(full_url: str) -> str:
    # url_clean = sanitize_url_for_parse(decode_layers(full_url, rounds=3))

    try:
        u = urlsplit(full_url)
    except ValueError:
        return decode_layers(full_url, rounds=3)

    if not (u.scheme and u.netloc):
        return decode_layers(full_url, rounds=3)

    q = parse_qsl(u.query, keep_blank_values=True)

    # collect values; decode each value (handles nested encoding inside values)
    values = [decode_layers(v, rounds=3) for _, v in q]
    core = " ".join(v for v in values if v).strip()

    if len(core) <= 0:
      # payload in last url path
      u_path = u.path.split('/')[-1]
      core = decode_layers(u_path, rounds=3).strip()

    if u.fragment:
        core = (core + " " + decode_layers(u.fragment, rounds=3)).strip()

    return core #if core else decode_layers(url_clean, rounds=3)

def preprocess(full_url: str) -> str:
    s = url_to_payload_core(full_url)
    s = s.lower()
    s = s.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


# =======================
# Feature extraction (binary presence)
# =======================
def extract_features(s: str) -> Dict[str, int]:
    feats: Dict[str, int] = {}

    # ---- Table 1 (structural) ----
    for ch in PUNCTUATION_CHARS:
        feats[safe_feature_name(ch, "punc")] = 1 if ch in s else 0

    for combo in PUNCTUATION_COMBOS:
        feats[safe_feature_name(combo, "combo")] = 1 if combo in s else 0

    # ---- Table 2 (keyword) ----
    for name, pat in WORD_PATTERNS:
        feats[name] = 1 if pat.search(s) else 0

    for name, lit in LITERAL_PATTERNS:
        feats[name] = 1 if lit in s else 0

    return feats

# =======================
# Calculate Readability Feature
# =======================


# ---------- Readability heuristic ----------
VOWELS = set("aeiouAEIOU")
TOKEN_SPLIT_RE = re.compile(r"[^A-Za-z0-9]+")  # bạn đã có ở phần trước thì khỏi khai báo lại

def tokenize_payload_readability(payload: str):
    return [t for t in TOKEN_SPLIT_RE.split(payload) if t]

def payload_human_readable_by_percentage(payload: str) -> int:
    """
    Trả về 1 nếu payload 'giống văn bản người đọc' theo các ngưỡng heuristic,
    ngược lại trả 0. (dạng int để làm feature)
    """
    words = tokenize_payload_readability(payload)
    total = len(words)
    if total == 0:
        return 0

    alpha_ok = vowel_ok = length_ok = repetition_ok = 0

    # ceil thresholds
    need_alpha  = int(0.45 * total + 0.999999)
    need_vowel  = int(0.40 * total + 0.999999)
    need_length = int(0.70 * total + 0.999999)
    need_rep    = int(0.80 * total + 0.999999)

    for i, w in enumerate(words, start=1):
        n_total = len(w)

        if n_total < 15:
            length_ok += 1

        # 1-pass scan: count alpha + vowels + detect >=3 consecutive
        n_alpha = 0
        n_vowel = 0
        rep_bad = False

        prev = None
        run = 0

        for ch in w:
            # consecutive repetition
            if ch == prev:
                run += 1
                if run >= 3:
                    rep_bad = True
            else:
                prev = ch
                run = 1

            # ASCII alpha (consistent with your tokenization A-Za-z0-9)
            if ('A' <= ch <= 'Z') or ('a' <= ch <= 'z'):
                n_alpha += 1
                if ch in VOWELS:
                    n_vowel += 1

        if not rep_bad:
            repetition_ok += 1

        if n_total > 0 and (n_alpha / n_total) > 0.70:
            alpha_ok += 1

        if n_alpha > 0:
            vr = n_vowel / n_alpha
            if 0.20 < vr < 0.60:
                vowel_ok += 1

        # early exit
        remaining = total - i
        if alpha_ok + remaining < need_alpha:
            return 0
        if vowel_ok + remaining < need_vowel:
            return 0
        if length_ok + remaining < need_length:
            return 0
        if repetition_ok + remaining < need_rep:
            return 0

    return int(
        alpha_ok >= need_alpha and
        vowel_ok >= need_vowel and
        length_ok >= need_length and
        repetition_ok >= need_rep
    )


# =======================
# Main run (Colab cell)
# =======================
df = read_csv_robust(INPUT_CSV)

# Map Class -> label (0/1)
df["label"] = df[CLASS_COL].astype(str).str.strip().str.lower().map({
    "malicious": 1,
    "benign": 0,
})


# Preprocess URLs into payload core
text_series = df[TEXT_COL].map(preprocess)
feat_df = pd.DataFrame(list(text_series.map(extract_features)))

# Readability feature (0/1)
readability = text_series.map(payload_human_readable_by_percentage)

# Merge + Save
out = pd.concat([feat_df, readability.rename("readability"), df["label"]], axis=1)
out.to_csv(OUTPUT_CSV, index=False)

print("Saved:", OUTPUT_CSV)
print("Rows:", out.shape[0], "| Added feature cols:", feat_df.shape[1])
out.head()


Saved: /content/drive/MyDrive/ATPM/features.csv
Rows: 42671 | Added feature cols: 58


,punc_amp,punc_pct,punc_slash,punc_bslash,punc_plus,punc_squote,punc_qmark,punc_emark,punc_semi,punc_hash,punc_eq,punc_lbrack,punc_rbrack,punc_dollar,punc_lparen,punc_rparen,punc_caret,punc_star,punc_comma,punc_dash,punc_lt,punc_gt,punc_at,punc_underscore,punc_colon,punc_lbrace,punc_rbrace,punc_tilde,punc_dot,punc_space,punc_pipe,punc_brokenbar,punc_dquote,combo_gt_lt,combo_squote_dquote_gt_lt,combo_lbrack_rbrack,combo_eq_eq,combo_amp_hash,obj_document,obj_window,obj_iframe,obj_location,obj_this,evt_onload,evt_onerror,mth_createelement,mth_fromcharcode,mth_search,attr_src,attr_href,attr_cookie,kw_var,proto_http,tag_div,tag_img,tag_script,fn_eval,ext_js,readability,label
0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,1,1,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,1
1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,1,1,0,0,0,0,0,0,1,1,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1
2,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1
3,0,0,1,1,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,1,1,0,0,0,0,0,0,1,1,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,1
4,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,1,1,0,0,0,0,0,0,1,1,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1


In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/ATPM/features.csv")
pd.set_option("display.max_columns", None)
df.head()

,punc_amp,punc_pct,punc_slash,punc_bslash,punc_plus,punc_squote,punc_qmark,punc_emark,punc_semi,punc_hash,punc_eq,punc_lbrack,punc_rbrack,punc_dollar,punc_lparen,punc_rparen,punc_caret,punc_star,punc_comma,punc_dash,punc_lt,punc_gt,punc_at,punc_underscore,punc_colon,punc_lbrace,punc_rbrace,punc_tilde,punc_dot,punc_space,punc_pipe,punc_brokenbar,punc_dquote,combo_gt_lt,combo_squote_dquote_gt_lt,combo_lbrack_rbrack,combo_eq_eq,combo_amp_hash,obj_document,obj_window,obj_iframe,obj_location,obj_this,evt_onload,evt_onerror,mth_createelement,mth_fromcharcode,mth_search,attr_src,attr_href,attr_cookie,kw_var,proto_http,tag_div,tag_img,tag_script,fn_eval,ext_js,label
0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,1,1,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1
1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,1,1,0,0,0,0,0,0,1,1,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1
2,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
3,0,0,1,1,0,0,0,0,1,0,1,0,0,0,1,1,0,0,0,0,1,1,0,0,0,0,0,0,1,1,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1
4,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,1,1,0,0,0,0,0,0,1,1,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.svm import NuSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix


# ---------------- Config ----------------
DATA_PATH = "/content/drive/MyDrive/ATPM/features.csv"
N_SPLITS = 5
RANDOM_STATE = 42
# ----------------------------------------


# =======================
# Load dataset
# =======================
df = pd.read_csv(DATA_PATH, encoding="latin1")
assert "label" in df.columns

y = df["label"].astype(int)

drop_cols = ["label"]
for c in ["Payloads", "Payloads_normalized", "Class"]:
    if c in df.columns:
        drop_cols.append(c)

X = df.drop(columns=drop_cols)

print("Loaded:", X.shape, "| Labels:", y.shape)


# =======================
# Define models
# =======================
models = {
    "LogReg (L2, C=1.0)": Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(
            C=1.0,
            penalty="l2",
            solver="liblinear",   # ổn cho binary + dữ liệu vừa/nhỏ
            max_iter=5000,
            random_state=RANDOM_STATE
        ))
    ]),
    # "SVM linear (C=7)": Pipeline([
    #     ("scaler", StandardScaler(with_mean=True)),
    #     ("clf", SVC(
    #         kernel='linear'
    #     ))
    # ])
    # "SVM Poly (deg=3,nu=0.10)": Pipeline([
    #     ("scaler", StandardScaler(with_mean=False)),
    #     ("clf", NuSVC(
    #       kernel="poly",
    #       degree=3,
    #       nu=0.10,
    #       gamma="scale"
    #     ))
    # ])
    "KNN (k=1)": Pipeline([
        ("scaler", StandardScaler(with_mean=True)),
        ("clf", KNeighborsClassifier(n_neighbors=1))
    ])
    # "Random Forest (n=40)": RandomForestClassifier(
    #     n_estimators=40,
    #     bootstrap=True,
    #     oob_score=True,
    #     n_jobs=-1,
    #     random_state=RANDOM_STATE
    # )
}

# =======================
# Metric helper
# =======================
def metrics_from_cm(cm):
    TN, FP, FN, TP = cm.ravel()

    # Sensitive / Recall (có lọt payload độc không) cao → ít lọt mã độc
    # Specificity (Có chặn nhầm benign không) cao → ít block nhầm
    # Precision (Báo độc có đáng tin không) cao → ít báo động giả
    # F1 cao → mô hình cân bằng tốt

    acc = (TP + TN) / (TP + TN + FP + FN) # Độ chính xác
    sens = TP / (TP + FN) if (TP + FN) else 0 # Recall / True Positive Rate
    spec = TN / (TN + FP) if (TN + FP) else 0 # True Negative Rate
    prec = TP / (TP + FP) if (TP + FP) else 0 # Độ tin cậy
    f1 = (2 * prec * sens / (prec + sens)) if (prec + sens) else 0

    return acc, sens, spec, prec, f1

def labeled_cm_df(cm):
    return pd.DataFrame(
        cm,
        index=["Actual Negative(0)", "Actual Positive(1)"],
        columns=["Predicted Negative(0)", "Predicted Positive(1)"]
    )

# =======================
# 5-Fold CV splits
# =======================
kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
splits = list(kf.split(X, y))

# Store pooled confusion matrices for all models
final_confusions = {}


# =======================
# Run CV per model
# =======================
for model_name, model in models.items():

    print("\n" + "="*80)
    print("MODEL:", model_name)
    print("="*80)

    fold_scores = []
    all_true, all_pred = [], []

    for fold_idx, (train_idx, test_idx) in enumerate(splits, start=1):

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
        acc, sens, spec, prec, f1 = metrics_from_cm(cm)

        fold_scores.append((acc, sens, spec, prec, f1))

        all_true.append(y_test.to_numpy())
        all_pred.append(y_pred)

        print(f"Fold {fold_idx}: "
              f"Acc={acc:.4f} Sens={sens:.4f} Spec={spec:.4f} "
              f"Prec={prec:.4f} F1={f1:.4f}")

    # Mean ± Std
    arr = np.array(fold_scores)
    means = arr.mean(axis=0)
    stds = arr.std(axis=0)

    print("-"*80)
    print(f"MEAN±STD: "
          f"Acc={means[0]:.4f}±{stds[0]:.4f}  "
          f"Sens={means[1]:.4f}±{stds[1]:.4f}  "
          f"Spec={means[2]:.4f}±{stds[2]:.4f}  "
          f"Prec={means[3]:.4f}±{stds[3]:.4f}  "
          f"F1={means[4]:.4f}±{stds[4]:.4f}")

    # Pooled confusion matrix (all folds combined)
    y_true_all = np.concatenate(all_true)
    y_pred_all = np.concatenate(all_pred)

    cm_all = confusion_matrix(y_true_all, y_pred_all, labels=[0, 1])
    final_confusions[model_name] = labeled_cm_df(cm_all)


# =======================
# Print Confusion Matrices ONCE at the end
# =======================
# Thực tế \ Dự đoán	  Benign (0)	Malicious (1)
# Benign (0)	         TN	          FP
# Malicious (1)	       FN	          TP

print("\n\n" + "#"*90)
print("CONFUSION MATRICES (POOLED over 5 folds) - Printed Once for All Models")
print("#"*90)

for model_name, cm in final_confusions.items():
    print("\nModel:", model_name)
    display(final_confusions[model_name])


Loaded: (42671, 59) | Labels: (42671,)

MODEL: LogReg (L2, C=1.0)
Fold 1: Acc=0.9837 Sens=0.9699 Spec=0.9909 Prec=0.9823 F1=0.9761
Fold 2: Acc=0.9843 Sens=0.9692 Spec=0.9922 Prec=0.9847 F1=0.9769
Fold 3: Acc=0.9828 Sens=0.9706 Spec=0.9891 Prec=0.9789 F1=0.9747
Fold 4: Acc=0.9839 Sens=0.9692 Spec=0.9916 Prec=0.9837 F1=0.9764
Fold 5: Acc=0.9869 Sens=0.9750 Spec=0.9931 Prec=0.9865 F1=0.9807
--------------------------------------------------------------------------------
MEAN±STD: Acc=0.9843±0.0014  Sens=0.9708±0.0022  Spec=0.9914±0.0013  Prec=0.9832±0.0025  F1=0.9769±0.0020

MODEL: KNN (k=1)
Fold 1: Acc=0.9228 Sens=0.9921 Spec=0.8867 Prec=0.8200 F1=0.8979
Fold 2: Acc=0.9289 Sens=0.9904 Spec=0.8968 Prec=0.8332 F1=0.9051
Fold 3: Acc=0.9250 Sens=0.9921 Spec=0.8901 Prec=0.8245 F1=0.9006
Fold 4: Acc=0.9181 Sens=0.9894 Spec=0.8810 Prec=0.8122 F1=0.8921
Fold 5: Acc=0.9173 Sens=0.9928 Spec=0.8780 Prec=0.8089 F1=0.8915
-------------------------------------------------------------------------------

,Predicted Negative(0),Predicted Positive(1)
Actual Negative(0),27826,242
Actual Positive(1),427,14176



Model: KNN (k=1)


,Predicted Negative(0),Predicted Positive(1)
Actual Negative(0),24883,3185
Actual Positive(1),126,14477
